In [0]:
RAW_TABLE = "workspace.default.sample_superstore_raw"

TABLE_PATH = "/Volumes/workspace/default/delta_assignment/tables"
SCD1_TABLE = f"{TABLE_PATH}/customer_scd1"
SCD2_TABLE = f"{TABLE_PATH}/customer_scd2"
TRACKED_COLS = ["segment", "city", "state", "region", "total_spent"]
SPLIT_QUANTILE = 0.85

print(f"Reading from table: {RAW_TABLE}")
print(f"Writing Delta tables to: {TABLE_PATH}")

Reading from table: workspace.default.sample_superstore_raw
Writing Delta tables to: /Volumes/workspace/default/delta_assignment/tables


In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

def _reset_path(path):
    try:
        dbutils.fs.rm(path, recurse=True)
    except Exception:
        pass  # nothing to remove yet — fine on a first run

_reset_path(SCD1_TABLE)
_reset_path(SCD2_TABLE)
print("Environment ready.")

Environment ready.


In [0]:
raw_orders = spark.table(RAW_TABLE)
print(f"Raw Superstore rows: {raw_orders.count()}")
print(f"Unique customers: {raw_orders.select('Customer ID').distinct().count()}")
print(f"Order Date dtype: {dict(raw_orders.dtypes)['Order Date']}")
display(raw_orders.limit(10))

Raw Superstore rows: 9994
Unique customers: 793
Order Date dtype: date


Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0.0,14.1694
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0.0,1.9656
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0.0,34.47


In [0]:
orders = raw_orders.withColumnRenamed("Order Date", "order_date")

w_cum = (
    Window.partitionBy("Customer ID")
    .orderBy("order_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

snapshots = (
    orders
    .withColumn("total_spent", F.round(F.sum("Sales").over(w_cum), 2))
    .select(
        F.col("Customer ID").alias("customer_id"),
        F.col("Customer Name").alias("name"),
        F.col("Segment").alias("segment"),
        F.col("City").alias("city"),
        F.col("State").alias("state"),
        F.col("Region").alias("region"),
        F.col("Postal Code").alias("postal_code"),
        "order_date",
        "total_spent",
    )
    .withColumnRenamed("order_date", "snapshot_date")
)

cutoff_ts = snapshots.select(
    F.percentile_approx(F.unix_timestamp("snapshot_date"), SPLIT_QUANTILE)
).first()[0]

master_snapshots = snapshots.filter(F.unix_timestamp("snapshot_date") < cutoff_ts)
incremental_snapshots = snapshots.filter(F.unix_timestamp("snapshot_date") >= cutoff_ts)

print(f"Master (pre-cutoff) rows: {master_snapshots.count()} ({master_snapshots.select('customer_id').distinct().count()} customers)")
print(f"Incremental (post-cutoff) rows: {incremental_snapshots.count()}")

Master (pre-cutoff) rows: 8489 (789 customers)
Incremental (post-cutoff) rows: 1505


In [0]:
w_latest = Window.partitionBy("customer_id").orderBy(F.col("snapshot_date").desc())
incremental_df = (
    incremental_snapshots
    .withColumn("_rn", F.row_number().over(w_latest))
    .filter("_rn = 1")
    .drop("_rn")
)

new_ids_preview = incremental_df.join(master_snapshots.select("customer_id").distinct(), "customer_id", "left_anti").count()
print(f"customer_incremental: {incremental_df.count()} rows ({incremental_df.count() - new_ids_preview} updates, {new_ids_preview} new)")
display(incremental_df.limit(10))

customer_incremental: 494 rows (490 updates, 4 new)


customer_id,name,segment,city,state,region,postal_code,snapshot_date,total_spent
AA-10375,Allen Armold,Consumer,New York City,New York,East,10035,2017-12-11,921.47
AA-10645,Anna Andreadi,Consumer,San Diego,California,West,92105,2017-11-05,5086.94
AB-10060,Adam Bellavance,Home Office,Seattle,Washington,West,98105,2017-11-06,7197.09
AB-10105,Adrian Barton,Consumer,Henderson,Kentucky,South,42420,2017-11-19,14451.61
AB-10150,Aimee Bixby,Consumer,Carrollton,Texas,Central,75007,2017-11-19,844.91
AB-10165,Alan Barnes,Consumer,Bellevue,Washington,West,98006,2017-12-05,1098.86
AB-10600,Ann Blume,Corporate,Tucson,Arizona,West,85705,2017-11-10,79.2
AC-10450,Amy Cox,Consumer,Lafayette,Louisiana,South,70506,2017-12-19,5527.85
AC-10615,Ann Chong,Corporate,Rochester,New York,East,14609,2017-12-24,2537.69
AD-10180,Alan Dominguez,Home Office,Fairfield,Connecticut,East,6824,2017-12-01,5209.73


In [0]:
raw_df = (
    master_snapshots
    .withColumn("segment", F.when(F.rand(42) < 0.015, F.lit(None)).otherwise(F.col("segment")))
    .withColumn("city", F.when(F.rand(43) < 0.01, F.lit(None)).otherwise(F.col("city")))
    .withColumn("snapshot_date", F.col("snapshot_date").cast("string"))
)
incremental_df = incremental_df.withColumn("snapshot_date", F.col("snapshot_date").cast("string"))

print(f"customer_master row count: {raw_df.count()}")
display(raw_df.limit(20))

customer_master row count: 8489


customer_id,name,segment,city,state,region,postal_code,snapshot_date,total_spent
AA-10315,Alex Avila,Consumer,San Francisco,California,West,94122,2014-03-31,673.57
AA-10315,Alex Avila,Consumer,San Francisco,California,West,94122,2014-03-31,726.55
AA-10315,Alex Avila,Consumer,New York City,New York,East,10011,2014-09-15,741.49
AA-10315,Alex Avila,Consumer,New York City,New York,East,10011,2014-09-15,756.05
AA-10315,Alex Avila,null,San Francisco,California,West,94109,2015-10-04,783.01
AA-10315,Alex Avila,Consumer,Round Rock,Texas,Central,78664,2016-03-03,4713.08
AA-10315,Alex Avila,Consumer,Round Rock,Texas,Central,78664,2016-03-03,4715.38
AA-10315,Alex Avila,Consumer,Round Rock,Texas,Central,78664,2016-03-03,5147.36
AA-10315,Alex Avila,Consumer,Round Rock,Texas,Central,78664,2016-03-03,5189.08
AA-10315,Alex Avila,Consumer,Minneapolis,Minnesota,Central,55407,2017-06-29,5552.02


In [0]:
raw_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.customer_scd1")

dt = DeltaTable.forName(spark, "workspace.default.customer_scd1")
loaded = dt.toDF()
print(f"Delta table created at '{SCD1_TABLE}', version {dt.history(1).select('version').first()[0]}")
print(f"Row count: {loaded.count()}")
print(f"Duplicate customer_id rows: {loaded.count() - loaded.dropDuplicates(['customer_id']).count()}")
print(f"Null values: {sum(loaded.filter(F.col(c).isNull()).count() for c in loaded.columns)}")

Delta table created at '/Volumes/workspace/default/delta_assignment/tables/customer_scd1', version 8
Row count: 8489
Duplicate customer_id rows: 7700
Null values: 227


In [0]:
current = DeltaTable.forName(spark, "workspace.default.customer_scd1").toDF()
print("Nulls per column before cleaning:")
current.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in current.columns]).show()
print(f"Duplicate customer_id rows before cleaning: {current.count() - current.dropDuplicates(['customer_id']).count()}")

Nulls per column before cleaning:
+-----------+----+-------+----+-----+------+-----------+-------------+-----------+
|customer_id|name|segment|city|state|region|postal_code|snapshot_date|total_spent|
+-----------+----+-------+----+-----+------+-----------+-------------+-----------+
|          0|   0|    143|  84|    0|     0|          0|            0|          0|
+-----------+----+-------+----+-----+------+-----------+-------------+-----------+

Duplicate customer_id rows before cleaning: 7700


In [0]:
w = Window.partitionBy("customer_id").orderBy(F.col("snapshot_date").desc())
clean_df = (
    current
    .withColumn("_rn", F.row_number().over(w))
    .filter("_rn = 1")
    .drop("_rn")
    .fillna({"segment": "Unknown", "city": "Unknown City"})
)

clean_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.customer_scd1")

cleaned = DeltaTable.forName(spark, "workspace.default.customer_scd1").toDF()
print(f"Row count after cleaning: {cleaned.count()}")
print(f"Remaining duplicate customer_id rows: {cleaned.count() - cleaned.dropDuplicates(['customer_id']).count()}")
display(cleaned.orderBy("customer_id").limit(20))

Row count after cleaning: 789
Remaining duplicate customer_id rows: 0


customer_id,name,segment,city,state,region,postal_code,snapshot_date,total_spent
AA-10315,Alex Avila,Consumer,Minneapolis,Minnesota,Central,55407,2017-06-29,5552.02
AA-10375,Allen Armold,Consumer,Providence,Rhode Island,East,2908,2017-09-07,866.56
AA-10480,Andrew Allen,Consumer,Concord,North Carolina,South,28027,2017-04-15,1790.51
AA-10645,Anna Andreadi,Consumer,Georgetown,Kentucky,South,40324,2016-09-04,4824.84
AB-10015,Aaron Bergman,Consumer,Oklahoma City,Oklahoma,Central,73120,2016-11-10,544.2
AB-10060,Adam Bellavance,Home Office,Los Angeles,California,West,90004,2017-05-07,4899.35
AB-10105,Adrian Barton,Consumer,Bloomington,Illinois,Central,61701,2017-08-03,13037.42
AB-10150,Aimee Bixby,Consumer,Long Beach,New York,East,11561,2017-09-04,828.01
AB-10165,Alan Barnes,Consumer,Toledo,Ohio,East,43615,2017-04-14,806.36
AB-10255,Alejandro Ballentine,Home Office,Los Angeles,California,West,90036,2017-07-17,914.53


In [0]:
scd2_init = (
    cleaned
    .withColumn("effective_start_date", F.lit("2014-01-01"))
    .withColumn("effective_end_date", F.lit(None).cast("string"))
    .withColumn("is_current", F.lit(True))
)
scd2_init.write.format("delta").mode("overwrite").saveAsTable("workspace.default.customer_scd2")
print(f"SCD2 table initialized with {DeltaTable.forName(spark, 'workspace.default.customer_scd2').toDF().count()} rows (all is_current = True)")

SCD2 table initialized with 789 rows (all is_current = True)


In [0]:
scd1_table = DeltaTable.forName(spark, "workspace.default.customer_scd1")
v_before = scd1_table.history(1).select("version").first()[0]

(
    scd1_table.alias("target")
    .merge(incremental_df.alias("source"), "target.customer_id = source.customer_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

v_after = scd1_table.history(1).select("version").first()[0]
print(f"SCD1 table version before merge: {v_before}")
print(f"SCD1 table version after merge:  {v_after}")
display(scd1_table.history(1).select("operation", "operationMetrics"))

SCD1 table version before merge: 9
SCD1 table version after merge:  10


operation,operationMetrics
MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 15106, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 490, executionTimeMs -> 6793, materializeSourceTimeMs -> 866, numTargetRowsInserted -> 4, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3236, numTargetRowsUpdated -> 490, numOutputRows -> 494, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 494, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2580)"


In [0]:
scd2_table = DeltaTable.forName(spark, "workspace.default.customer_scd2")
current_now = scd2_table.toDF().filter("is_current = true")

old_cols = [F.col(c).alias(f"{c}_old") for c in TRACKED_COLS]
compare = incremental_df.join(current_now.select("customer_id", *old_cols), on="customer_id", how="left")

change_condition = F.col(f"{TRACKED_COLS[0]}_old").isNull()
for c in TRACKED_COLS:
    change_condition = change_condition | (F.col(c) != F.col(f"{c}_old"))

changed_df = compare.filter(change_condition).select("customer_id").distinct()
changed_ids_count = changed_df.count()
print(f"Customers needing a new SCD2 version: {changed_ids_count} / {incremental_df.count()}")

Customers needing a new SCD2 version: 494 / 494


In [0]:
scd1_final = DeltaTable.forName(spark, "workspace.default.customer_scd1").toDF()
new_ids_count = incremental_df.join(cleaned, "customer_id", "left_anti").count()
expected_scd1_rows = cleaned.count() + new_ids_count
dup_count_scd1 = scd1_final.count() - scd1_final.dropDuplicates(["customer_id"]).count()

print("SCD1 VALIDATION")
print(f"Row count check: {'PASS' if scd1_final.count() == expected_scd1_rows else 'FAIL'}")
print(f"Duplicate check: {'PASS' if dup_count_scd1 == 0 else 'FAIL'}")

scd2_final = DeltaTable.forName(spark, "workspace.default.customer_scd2").toDF()
expected_scd2_rows = cleaned.count() + changed_ids_count
current_rows = scd2_final.filter("is_current = true")
dup_current = current_rows.count() - current_rows.dropDuplicates(["customer_id"]).count()

print("SCD2 VALIDATION")
print(f"Row count check: {'PASS' if scd2_final.count() == expected_scd2_rows else 'FAIL'}")
print(f"Exactly-one-current-row check: {'PASS' if dup_current == 0 else 'FAIL'}")

SCD1 VALIDATION
Row count check: PASS
Duplicate check: PASS
SCD2 VALIDATION
Row count check: FAIL
Exactly-one-current-row check: PASS


In [0]:
print(f"FINAL SCD1 TABLE — {scd1_final.count()} rows")
display(scd1_final.orderBy("customer_id"))

FINAL SCD1 TABLE — 793 rows


customer_id,name,segment,city,state,region,postal_code,snapshot_date,total_spent
AA-10315,Alex Avila,Consumer,Minneapolis,Minnesota,Central,55407,2017-06-29,5552.02
AA-10375,Allen Armold,Consumer,New York City,New York,East,10035,2017-12-11,921.47
AA-10480,Andrew Allen,Consumer,Concord,North Carolina,South,28027,2017-04-15,1790.51
AA-10645,Anna Andreadi,Consumer,San Diego,California,West,92105,2017-11-05,5086.94
AB-10015,Aaron Bergman,Consumer,Oklahoma City,Oklahoma,Central,73120,2016-11-10,544.2
AB-10060,Adam Bellavance,Home Office,Seattle,Washington,West,98105,2017-11-06,7197.09
AB-10105,Adrian Barton,Consumer,Henderson,Kentucky,South,42420,2017-11-19,14451.61
AB-10150,Aimee Bixby,Consumer,Carrollton,Texas,Central,75007,2017-11-19,844.91
AB-10165,Alan Barnes,Consumer,Bellevue,Washington,West,98006,2017-12-05,1098.86
AB-10255,Alejandro Ballentine,Home Office,Los Angeles,California,West,90036,2017-07-17,914.53


In [0]:
print(f"FINAL SCD2 TABLE — {scd2_final.count()} rows ({current_rows.count()} current)")
display(
    scd2_final.select("customer_id", "name", "segment", "city", "state", "region",
        "total_spent", "effective_start_date", "effective_end_date", "is_current")
    .orderBy("customer_id", "effective_start_date")
)

FINAL SCD2 TABLE — 789 rows (789 current)


customer_id,name,segment,city,state,region,total_spent,effective_start_date,effective_end_date,is_current
AA-10315,Alex Avila,Consumer,Minneapolis,Minnesota,Central,5552.02,2014-01-01,null,true
AA-10375,Allen Armold,Consumer,Providence,Rhode Island,East,866.56,2014-01-01,null,true
AA-10480,Andrew Allen,Consumer,Concord,North Carolina,South,1790.51,2014-01-01,null,true
AA-10645,Anna Andreadi,Consumer,Georgetown,Kentucky,South,4824.84,2014-01-01,null,true
AB-10015,Aaron Bergman,Consumer,Oklahoma City,Oklahoma,Central,544.2,2014-01-01,null,true
AB-10060,Adam Bellavance,Home Office,Los Angeles,California,West,4899.35,2014-01-01,null,true
AB-10105,Adrian Barton,Consumer,Bloomington,Illinois,Central,13037.42,2014-01-01,null,true
AB-10150,Aimee Bixby,Consumer,Long Beach,New York,East,828.01,2014-01-01,null,true
AB-10165,Alan Barnes,Consumer,Toledo,Ohio,East,806.36,2014-01-01,null,true
AB-10255,Alejandro Ballentine,Home Office,Los Angeles,California,West,914.53,2014-01-01,null,true


In [0]:
print("RUN SUMMARY")
print("=" * 55)
print(f"Raw Superstore orders read:      {raw_orders.count()}")
print(f"Master baseline (after cleaning): {cleaned.count()} customers")
print(f"Incremental batch:                {incremental_df.count()} rows "
      f"({changed_ids_count - new_ids_count} updates, {new_ids_count} new)")
print(f"SCD1 final rows:                  {scd1_final.count()}  (history not kept)")
print(f"SCD2 final rows:                  {scd2_final.count()}  "
      f"({current_rows.count()} current + {scd2_final.count() - current_rows.count()} historical)")
print(f"SCD1 duplicate keys:              {dup_count_scd1}")
print(f"SCD2 duplicate current keys:      {dup_current}")
print("=" * 55)

RUN SUMMARY
Raw Superstore orders read:      9994
Master baseline (after cleaning): 793 customers
Incremental batch:                494 rows (494 updates, 0 new)
SCD1 final rows:                  793  (history not kept)
SCD2 final rows:                  789  (789 current + 0 historical)
SCD1 duplicate keys:              0
SCD2 duplicate current keys:      0
